# VaR Backtesting and Stress Testing

Module: Market Risk

## Lesson summary

This lab extends VaR estimation into model validation. Students generate rolling VaR forecasts, test exception frequency and exception clustering, map recent exceptions to the Basel traffic-light logic, and complement probabilistic risk metrics with deterministic stress scenarios.

## Learning objectives

By the end of this lab, students should be able to:

- build a rolling one-day VaR forecast;
- identify VaR exceptions from realized portfolio returns;
- run Kupiec unconditional coverage and Christoffersen independence tests;
- interpret a Basel traffic-light result;
- design a deterministic stress test for a multi-asset portfolio.

## Backtesting equations

A VaR exception occurs when the realized loss exceeds the forecast threshold:

$$
I_t=\mathbf{1}\{-r_t>\widehat{\operatorname{VaR}}_{\alpha,t}\}.
$$

Under a correctly calibrated one-day VaR model, the expected exception rate is approximately $\alpha$:

$$
\mathbb{E}[I_t]=\alpha.
$$

Stress testing applies a deterministic shock vector $s$ to portfolio weights $w$:

$$
\operatorname{StressLoss}= -w^\top s.
$$

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

from src.market_risk import (
    basel_traffic_light,
    conditional_coverage_test,
    exception_series,
    stress_scenario_loss,
)

## Synthetic portfolio returns with regime changes

In [ ]:
rng = np.random.default_rng(41)
dates = pd.bdate_range("2022-01-03", periods=900)

low_volatility = rng.normal(0.00035, 0.009, size=350)
stress_volatility = rng.standard_t(df=5, size=200) * 0.018 - 0.0006
normalization = rng.normal(0.00025, 0.011, size=350)
portfolio_returns = pd.Series(
    np.concatenate([low_volatility, stress_volatility, normalization]),
    index=dates,
    name="portfolio_return",
)

portfolio_returns.describe()

## Rolling VaR forecasts

Historical VaR uses only information available before the realized return. The `shift(1)` is essential because today's return cannot be used to forecast today's risk.

In [ ]:
alpha = 0.01
window = 250

historical_var_forecast = (
    -portfolio_returns.rolling(window).quantile(alpha).shift(1).rename("historical_var")
)

rolling_mean = portfolio_returns.rolling(window).mean().shift(1)
rolling_volatility = portfolio_returns.rolling(window).std().shift(1)
gaussian_var_forecast = (
    -(rolling_mean + norm.ppf(alpha) * rolling_volatility).rename("gaussian_var")
)

pd.concat(
    [portfolio_returns, historical_var_forecast, gaussian_var_forecast],
    axis=1,
).dropna().head()

## Exception series

In [ ]:
historical_exceptions = exception_series(portfolio_returns, historical_var_forecast)
gaussian_exceptions = exception_series(portfolio_returns, gaussian_var_forecast)

pd.DataFrame(
    {
        "historical_exceptions": historical_exceptions.value_counts(),
        "gaussian_exceptions": gaussian_exceptions.value_counts(),
    }
).fillna(0).astype(int)

## Statistical backtesting

Kupiec's test checks whether the total number of exceptions is consistent with the chosen tail probability. Christoffersen's test checks whether exceptions are clustered.

In [ ]:
pd.DataFrame(
    {
        "historical_var": conditional_coverage_test(historical_exceptions, alpha=alpha),
        "gaussian_var": conditional_coverage_test(gaussian_exceptions, alpha=alpha),
    }
)

## Basel traffic-light view

The classic Basel traffic-light table is based on 250 trading days of exceptions. This example maps the latest 250 backtest observations to the zone and multiplier.

In [ ]:
latest_250_exception_count = int(historical_exceptions.tail(250).sum())
basel_traffic_light(latest_250_exception_count)

## Deterministic stress scenario

VaR and Expected Shortfall extrapolate from a probability model. Stress testing asks a different question: what happens if a specific macro-financial shock is imposed on the portfolio?

In [ ]:
weights = pd.Series(
    {
        "mexican_equity": 0.40,
        "global_equity": 0.30,
        "mxn_bond": 0.20,
        "usd_mxn_hedge": 0.10,
    },
    name="weight",
)

stress_shocks = pd.Series(
    {
        "mexican_equity": -0.40,
        "global_equity": -0.25,
        "mxn_bond": -0.08,
        "usd_mxn_hedge": 0.25,
    },
    name="shock",
)

stress_scenario_loss(weights, stress_shocks)

## Regulatory context

The Fundamental Review of the Trading Book moved market-risk capital from a VaR-centered framework toward Expected Shortfall. A practical implementation also has to account for liquidity horizons, because a position that takes 60 or 120 days to exit cannot be treated like a highly liquid 10-day risk factor.

For this course, the key modeling lesson is not to memorize a capital formula. The key lesson is that a backtested VaR model, an Expected Shortfall estimate, and a deterministic stress scenario answer different risk questions and should be documented together.

## Model limitations

- Backtests have low power when exceptions are rare, so passing a test is not proof that the risk model is reliable.
- Rolling windows trade responsiveness against estimation noise and can react slowly to abrupt regime changes.
- Stress scenarios are judgment-based and should complement, not replace, probabilistic risk estimates.